# 🛵 Food Delivery Analytics Challenge
**AI & DS — Beginner to Intermediate | Hackathon Task A**

This notebook loads, cleans, and analyzes 38,964 food-delivery records using
**Python & Pandas only** (no Machine Learning), answers the three competition
questions programmatically, produces two required visualizations, derives
business insights, and closes with an AI-generated plain-English explanation.

**Workflow:** Load → Clean → Analyze → Visualize → Interpret → Explain

> A polished, filterable version of everything below also lives in a
> **Streamlit dashboard** (`app.py`) included with this submission for the demo.


## 0. Setup

In [ ]:
# If you're running this in Google Colab, uncomment the line below the first time:
# !pip install -q pandas numpy matplotlib anthropic openai groq

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
print("Libraries loaded ✅")

### Get the dataset
Run **one** of the two cells below depending on where the CSV lives.


In [ ]:
# Option A — Google Colab: upload the CSV interactively
# from google.colab import files
# uploaded = files.upload()   # choose food_delivery_dataset.csv
# CSV_PATH = list(uploaded.keys())[0]

# Option B — CSV already sitting next to this notebook / in Colab's file panel
CSV_PATH = "food_delivery_dataset.csv"
print("Using:", CSV_PATH)

## A. Load & Understand the data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head()

In [ ]:
print("Column names:")
print(list(df.columns))

In [ ]:
df.dtypes

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
pd.DataFrame({"missing_values": missing, "missing_%": missing_pct}).query("missing_values > 0")

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## B. Clean the Data

**Cleaning decisions** (documented in full in `README.md`):

| Issue | Decision | Why |
|---|---|---|
| Whitespace in text columns (e.g. `"Jam "`) | Stripped | Prevents `"Jam"` and `"Jam "` being treated as different categories |
| Duplicate rows | Dropped | Avoids double-counting deliveries |
| Missing `Delivery_person_Age` | Filled with **median** | Numeric, roughly symmetric — median won't skew the average-age KPI |
| Missing `Delivery_person_Ratings` | Filled with **median** | Ratings are bounded 1–5 and left-skewed — median is more robust than mean |
| `Time_Orderd` in two encodings (clock string *and* Excel day-fraction float) | Both parsed into real datetimes | The raw file mixes formats — without this fix, thousands of valid timestamps look "missing" |
| `Time_Orderd` still missing after parsing | Left as `NaT`, excluded from time-of-day calcs | A true missing timestamp shouldn't be guessed |
| Distance ≤ 0 or delivery time ≤ 0 | Dropped (safety check) | Physically impossible values |
| Categorical columns | Cast to `category` dtype | Faster groupby, smaller memory footprint |


In [ ]:
def clean_data(df):
    df = df.copy()
    log = {"rows_before": len(df)}

    # strip whitespace on text/object columns
    text_cols = df.select_dtypes(include="object").columns.tolist()
    for c in text_cols:
        df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})

    log["duplicates_dropped"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    for c in ["Delivery_person_Age", "Delivery_person_Ratings", "Time_taken (min)",
              "distance_km", "multiple_deliveries", "Vehicle_condition"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    age_missing = int(df["Delivery_person_Age"].isna().sum())
    rating_missing = int(df["Delivery_person_Ratings"].isna().sum())
    age_med, rating_med = df["Delivery_person_Age"].median(), df["Delivery_person_Ratings"].median()
    df["Delivery_person_Age"] = df["Delivery_person_Age"].fillna(age_med)
    df["Delivery_person_Ratings"] = df["Delivery_person_Ratings"].fillna(rating_med)
    log.update(age_missing_filled=age_missing, age_fill_value=float(age_med),
               rating_missing_filled=rating_missing, rating_fill_value=float(rating_med))

    df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y", errors="coerce")

    def parse_time(series):
        s = series.astype(str).str.strip()
        as_clock = pd.to_datetime(s, format="%H:%M", errors="coerce")
        frac = pd.to_numeric(s, errors="coerce")
        as_fraction = pd.to_datetime((frac * 86400).round(), unit="s", origin="1970-01-01", errors="coerce")
        return as_clock.fillna(as_fraction)

    df["Time_Orderd_parsed"] = parse_time(df["Time_Orderd"])
    df["Time_Order_picked_parsed"] = parse_time(df["Time_Order_picked"])
    log["time_orderd_missing_left_as_na"] = int(df["Time_Orderd_parsed"].isna().sum())
    df["Order_Hour"] = df["Time_Orderd_parsed"].dt.hour

    for c in ["Weather_conditions", "Road_traffic_density", "Type_of_order",
              "Type_of_vehicle", "Festival", "City", "delivery_speed"]:
        df[c] = df[c].astype("category")

    before = len(df)
    df = df[(df["distance_km"] > 0) & (df["Time_taken (min)"] > 0)].reset_index(drop=True)
    log["impossible_rows_dropped"] = before - len(df)
    log["rows_after"] = len(df)
    return df, log


df_clean, clean_log = clean_data(df)
clean_log

## C. Basic Analysis

In [ ]:
df_clean["delivery_speed_kmph"] = df_clean["distance_km"] / (df_clean["Time_taken (min)"] / 60)

stats = {
    "Total deliveries": len(df_clean),
    "Avg delivery time (min)": round(df_clean["Time_taken (min)"].mean(), 2),
    "Min delivery time (min)": int(df_clean["Time_taken (min)"].min()),
    "Max delivery time (min)": int(df_clean["Time_taken (min)"].max()),
    "Avg distance (km)": round(df_clean["distance_km"].mean(), 2),
    "Avg speed (km/h)": round(df_clean["delivery_speed_kmph"].mean(), 2),
    "Avg delivery-person rating": round(df_clean["Delivery_person_Ratings"].mean(), 2),
    "Avg delivery-person age": round(df_clean["Delivery_person_Age"].mean(), 1),
}
for k, v in stats.items():
    print(f"{k:32s}: {v}")

## D. The 3 Competition Questions
All three are answered **programmatically** by grouping the cleaned dataframe — nothing below is hard-coded.

### Q1 — Which road traffic condition has the highest average delivery time?

In [ ]:
q1 = df_clean.groupby("Road_traffic_density", observed=True)["Time_taken (min)"].mean().sort_values(ascending=False).round(2)
print(q1)
print(f"\n➡️  Answer: '{q1.index[0]}' traffic has the highest average delivery time at {q1.iloc[0]} minutes.")

### Q2 — How does delivery distance affect delivery time?

In [ ]:
corr = df_clean["distance_km"].corr(df_clean["Time_taken (min)"])
bins = [0, 5, 10, 15, 20, np.inf]
labels = ["0-5 km", "5-10 km", "10-15 km", "15-20 km", "20+ km"]
df_clean["distance_bucket"] = pd.cut(df_clean["distance_km"], bins=bins, labels=labels)
by_bucket = df_clean.groupby("distance_bucket", observed=True)["Time_taken (min)"].mean().round(2)

print(f"Correlation(distance, delivery time) = {corr:.3f}\n")
print(by_bucket)
print(f"\n➡️  Answer: correlation is {corr:.3f} (positive) — delivery time rises as distance increases,"
      f" climbing from {by_bucket.iloc[0]} min (0-5km) to {by_bucket.iloc[-1]} min ({labels[-1]}).")

### Q3 — Which weather × traffic combination has the highest average delivery time?

In [ ]:
combo = df_clean.groupby(["Weather_conditions", "Road_traffic_density"], observed=True)["Time_taken (min)"] \
                 .mean().sort_values(ascending=False).round(2)
print(combo.head(5))
w, t = combo.index[0]
print(f"\n➡️  Answer: '{w}' weather + '{t}' traffic is slowest, averaging {combo.iloc[0]} minutes.")

## E. Visualizations

**Chart 1 — Average delivery time by traffic density (bar chart)**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
palette = {"Jam": "#E63946", "High": "#F4A261", "Medium": "#2A9D8F", "Low": "#457B9D"}
colors = [palette.get(k, "#6D6875") for k in q1.index]
bars = ax.bar(q1.index.astype(str), q1.values, color=colors, edgecolor="white")
ax.set_title("Average Delivery Time by Road Traffic Density", fontsize=13, fontweight="bold")
ax.set_xlabel("Road Traffic Density"); ax.set_ylabel("Average Delivery Time (minutes)")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.spines[["top", "right"]].set_visible(False)
for b, v in zip(bars, q1.values):
    ax.annotate(f"{v:.1f}", (b.get_x() + b.get_width()/2, v), ha="center", va="bottom", fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig("chart1_traffic_vs_time.png", dpi=150)
plt.show()

**Chart 2 — Delivery distance vs. delivery time (scatter plot)**

In [ ]:
plot_df = df_clean.sample(min(4000, len(df_clean)), random_state=42)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(plot_df["distance_km"], plot_df["Time_taken (min)"], alpha=0.25, s=14, color="#2A9D8F", edgecolors="none")
z = np.polyfit(df_clean["distance_km"], df_clean["Time_taken (min)"], 1)
xs = np.linspace(df_clean["distance_km"].min(), df_clean["distance_km"].max(), 100)
ax.plot(xs, np.poly1d(z)(xs), color="#E63946", linewidth=2, label="Trend line")
ax.set_title("Delivery Distance vs. Delivery Time", fontsize=13, fontweight="bold")
ax.set_xlabel("Distance (km)"); ax.set_ylabel("Delivery Time (minutes)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("chart2_distance_vs_time.png", dpi=150)
plt.show()

## F. Business Insights

In [ ]:
insights = []

worst_t, worst_v = q1.index[0], q1.iloc[0]
best_t, best_v = q1.index[-1], q1.iloc[-1]
insights.append((
    f"'{worst_t}' traffic adds ~{round(worst_v - best_v, 1)} minutes per delivery",
    f"Average time is {worst_v} min under '{worst_t}' traffic vs {best_v} min under '{best_t}' traffic.",
    "Pad ETAs dynamically during high-traffic windows and pre-position riders on jam-prone routes ahead of the rush."
))

insights.append((
    f"Delivery time rises steadily with distance (corr = {corr:.3f})",
    "Average time by distance band: " + ", ".join(f"{k} → {v} min" for k, v in by_bucket.items()),
    "Route longer orders to riders already positioned nearby, and scale the promised delivery window with distance instead of a flat estimate."
))

(w, t), worst_combo = combo.index[0], combo.iloc[0]
insights.append((
    f"'{w}' weather + '{t}' traffic is the worst combination",
    f"These conditions together average {worst_combo} minutes — the slowest of any combination in the data.",
    "When this combination is forecast, proactively message customers with realistic ETAs and temporarily boost rider incentives."
))

veh = df_clean.groupby("Vehicle_condition", observed=True)["Time_taken (min)"].mean().round(2)
insights.append((
    "Poorer vehicle condition is associated with slower deliveries",
    "Average delivery time by vehicle-condition score: " + ", ".join(f"{k}: {v} min" for k, v in veh.items()),
    f"Tie vehicle maintenance checks to rider performance incentives, since condition score {veh.idxmax()} sees the slowest deliveries."
))

for i, (title, finding, so_what) in enumerate(insights, 1):
    print(f"{i}. {title}\n   Finding: {finding}\n   So what: {so_what}\n")

## G. AI-Powered Explanation

Python/Pandas has already computed every number above. Below we hand those
**already-calculated results** to an LLM (Claude, OpenAI, or Groq — pick one)
purely to translate them into a short business narrative. The API key is
**never hard-coded** — it's read from an environment variable, entered securely
with `getpass` if you don't already have it exported.


In [ ]:
import os, json
from getpass import getpass

PROVIDER = "groq"   # "anthropic" | "openai" | "groq"
key_name = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "groq": "GROQ_API_KEY"}[PROVIDER]

# Prompts for a key only if one isn't already set as an environment variable,
# and only if this frontend supports interactive input (skipped safely otherwise).
if not os.environ.get(key_name):
    try:
        os.environ[key_name] = getpass(f"Enter your {PROVIDER} API key (input hidden): ")
    except Exception:
        print(f"⚠️ No {key_name} set and no interactive input available — "
              f"the AI-explanation cell below will show a placeholder instead of a live call. "
              f"Set the environment variable and rerun to get a real explanation.")

payload = {
    "overall_stats": stats,
    "avg_delivery_time_by_traffic": q1.to_dict(),
    "distance_vs_time_correlation": round(corr, 3),
    "avg_time_by_distance_bucket": by_bucket.to_dict(),
    "worst_weather_traffic_combos": {f"{a} + {b}": v for (a, b), v in combo.head(5).items()},
}

prompt = f"""You are a data analyst explaining results to a non-technical food-delivery
business audience. Below is JSON containing numbers ALREADY calculated with Python/Pandas.
Do not invent any new numbers — only explain the ones given.

DATA:
{json.dumps(payload, indent=2, default=str)}

Write:
1. A 2-3 sentence executive summary of overall delivery performance.
2. A short explanation (3-5 sentences) of what the traffic, distance, and weather findings mean for operations.
3. Two concrete, actionable recommendations for the business.

Keep it under 200 words, plain English, no jargon, no markdown headers."""

print("Prompt ready — run the next cell to call the model.")

In [ ]:
def call_anthropic(prompt):
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    resp = client.messages.create(model="claude-sonnet-4-5-20250929", max_tokens=500,
                                   messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text

def call_openai(prompt):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(model="gpt-4o-mini", max_tokens=500,
                                           messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content

def call_groq(prompt):
    from groq import Groq
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    resp = client.chat.completions.create(model="llama-3.1-8b-instant", max_tokens=500,
                                           messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content

callers = {"anthropic": call_anthropic, "openai": call_openai, "groq": call_groq}

if not os.environ.get(key_name):
    print("⚠️ No API key available — this is a placeholder, not a real AI response.\n"
          "Set an API key in the cell above (or as an environment variable) and rerun to get a live explanation.")
else:
    try:
        explanation = callers[PROVIDER](prompt)
        print(explanation)
    except Exception as e:
        print("⚠️ Could not reach the LLM API. Check your key/network and rerun.")
        print("Error:", e)

## Conclusion

- **38,964** delivery records were cleaned (whitespace, dual time formats, missing age/rating) with every decision documented above and in `README.md`.
- **Traffic**, **distance**, and **weather** all measurably affect delivery time, each answered directly from the data with no hard-coded numbers.
- Two required visualizations were produced, plus a heatmap and interactive filters are available in the companion **Streamlit dashboard** (`app.py`) for the live demo.
- An LLM turns the computed statistics into a short, decision-ready explanation for a non-technical audience.
